# conv-kernel-shape — ex2: construct a Conv2d weight tensor from a spec dict

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-kernel-shape`. Running the final beacon cell reports progress against the `CNN: Kernel shape (OC, IC, KH, KW)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Kernel shape (OC, IC, KH, KW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-kernel-shape`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-kernel-shape"
DD_SUBTOPIC = "CNN: Kernel shape (OC, IC, KH, KW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Construct a Conv2d weight from spec — quick refresher

A `nn.Conv2d(IC, OC, (KH, KW))` weight tensor has shape **`(OC, IC, KH, KW)`** — output channels first. To build a weight tensor with custom contents that drops into a Conv2d slot, you must:

1. Get the axis order right: `(OC, IC, KH, KW)`.
2. Use floating-point dtype (`torch.float32` by default) — Conv2d won't accept int tensors as weights.
3. Total scalars = `OC * IC * KH * KW`.

**Common spec.** Constructors take `(in_channels, out_channels, kernel_size)` — note the order is IC-then-OC for the constructor, but the resulting `weight` tensor is OC-first. Don't conflate these two argument orderings.

**Quick sanity probe.** After construction, verify `weight.shape == (OC, IC, KH, KW)` AND `nn.Conv2d(IC, OC, (KH, KW)).weight.shape == weight.shape` — both must agree exactly.

### Exercise 2 — construct a Conv2d weight tensor from a spec dict

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(OC, IC, KH, KW)` weight-shape convention to build a float32 weight tensor that matches the shape of `nn.Conv2d(in_channels=IC, out_channels=OC, kernel_size=(KH, KW)).weight`.
> Keywords: conv2d, weight-shape, construction, axis-order
> ```

**KCs targeted:** `conv-kernel-axis-order`, `conv-weight-construction`

Implement `ex2_build_conv_weight(spec)`. Given a dict like:

```python
spec = {
    'in_channels':  3,
    'out_channels': 16,
    'kernel_height': 5,
    'kernel_width':  3,
    'fill_value':    0.25,    # every weight entry set to this
}
```

Return a `torch.Tensor` of dtype `float32` and shape **`(OC, IC, KH, KW)`** filled with `spec['fill_value']`.

**The catch.** The constructor args use `(in_channels, out_channels, kernel_size)` order, but the resulting weight tensor is OC-first. Don't write a (IC, OC, KH, KW) tensor — that's the ConvTranspose2d layout and the shape-check against `nn.Conv2d(...).weight.shape` will fail.

**Use `t.full(...)` or `t.empty(...).fill_(...)`** — anything that produces a fp32 tensor of the right shape. Don't use `t.tensor([[...]])` literal — too tedious for arbitrary OC, IC.

The test instantiates a matching `nn.Conv2d` and confirms `weight.shape == module.weight.shape` exactly (the layout check), and `weight.numel() == OC * IC * KH * KW` (param count).

In [ ]:
def ex2_build_conv_weight(spec: dict) -> Tensor:
    OC = spec['out_channels']
    IC = spec['in_channels']
    KH = spec['kernel_height']
    KW = spec['kernel_width']
    return t.full((OC, IC, KH, KW), float(spec['fill_value']), dtype=t.float32)


<details><summary>Solution</summary>

```python
def ex2_build_conv_weight(spec: dict) -> Tensor:
    OC = spec['out_channels']
    IC = spec['in_channels']
    KH = spec['kernel_height']
    KW = spec['kernel_width']
    return t.full((OC, IC, KH, KW), float(spec['fill_value']), dtype=t.float32)
```

**Why `t.full(..., dtype=t.float32)`.** `t.full((shape,), 0)` would default to `int64` for an integer fill value, which `nn.Conv2d` won't accept. Explicit `dtype=t.float32` defends against int fill values too.

**The OC-first rule, in plain English.** `weight[oc, ic, kh, kw]` is 'the `(kh, kw)` tap of input-channel `ic` for output-channel `oc`'. Reading axis 0 first as OC matches how you'd describe 'pick filter 5 of 16, then pick its red-channel tap at position (2, 1)'.

**Why the constructor order is different.** `nn.Conv2d(in, out, k)` follows the linear-algebra convention `y = Wx` where `W` is described as 'how to map FROM input TO output' — in→out order. But once you HAVE the weight tensor, indexing it by filter (`weight[oc]`) is more useful than indexing by input channel, so PyTorch's storage layout puts OC first. Two different conventions; both serve their use case.

**ConvTranspose2d gotcha.** Loading a Conv2d's weight into a ConvTranspose2d's slot (without transposing axis 0 and 1) raises `size mismatch`. Conv2d = `(OC, IC, KH, KW)`; ConvTranspose2d = `(IC, OC, KH, KW)`. The test deliberately asserts they're DIFFERENT to catch the mix-up.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()